In [8]:
!pip install transformers torch sklearn pandas numpy imbalanced-learn

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [11]:
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW # Import AdamW from torch.optim
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt

In [10]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


Data Preparation

In [50]:
# Load data
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')
test_labels = pd.read_csv('/content/test_labels.csv')
val_df= pd.read_csv('/content/validation.csv')

# Define labels
LABELS = ['toxic', 'abusive', 'vulgar', 'menace', 'offense', 'bigotry']
NUM_LABELS = len(LABELS)

# Tokenizer (use multilingual BERT if validation data is truly diverse)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')  # English-focused

In [51]:
from sklearn.utils import resample
#Separating majority (non-toxic) and minority (toxic)
df_majority = df_train[df_train['toxic'] == 0]
df_minority = df_train[df_train['toxic'] == 1]

#Downsample the majority class
df_majority_downsampled = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

#Combining minority and downsampled majority
train_df = pd.concat([df_minority, df_majority_downsampled])

#Shuffle the combined data
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

Custom Dataset Class

In [52]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.FloatTensor(label)
        }

# Prepare data
train_texts = df_train['feedback_text'].values
train_labels = df_train[LABELS].values

val_texts = df_val['feedback_text'].values
val_labels = df_val[['toxic']].values  # Only toxic for validation

# Create datasets
train_dataset = ToxicDataset(train_texts, train_labels, tokenizer)
val_dataset = ToxicDataset(val_texts, val_labels, tokenizer)  # Will only use toxic label

Model Initialization with Class Weighting

In [53]:
# Calculate class weights for imbalance (toxic vs non-toxic)
toxic_counts = df_train['toxic'].value_counts()
class_weights = torch.tensor([toxic_counts[0]/toxic_counts[1]], device='cuda')

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
).to('cuda')

# Loss function with weighting for toxic/non-toxic
loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=class_weights)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training Loop

In [54]:
optimizer = AdamW(model.parameters(), lr=2e-5)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

for epoch in range(3):  # 3 epochs typically enough for BERT fine-tuning
    # Training
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {
            'input_ids': batch['input_ids'].to('cuda'),
            'attention_mask': batch['attention_mask'].to('cuda'),
            'labels': batch['labels'].to('cuda')
        }
        outputs = model(**inputs)
        logits = outputs.logits
        loss = loss_fct(logits, inputs['labels'])
        loss.backward()
        optimizer.step()

    # Validation (only toxic)
    model.eval()
    toxic_preds = []
    toxic_true = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                'input_ids': batch['input_ids'].to('cuda'),
                'attention_mask': batch['attention_mask'].to('cuda')
            }
            logits = model(**inputs).logits
            toxic_logits = logits[:, 0]  # First column is toxic
            toxic_preds.extend(torch.sigmoid(toxic_logits).cpu().numpy())
            toxic_true.extend(batch['labels'].cpu().numpy())

    # Calculate toxic-specific metrics
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(toxic_true, toxic_preds)
    print(f"Epoch {epoch+1} | Toxic Val AUC: {auc:.4f}")

Epoch 1 | Toxic Val AUC: 0.5750
Epoch 2 | Toxic Val AUC: 0.6470
Epoch 3 | Toxic Val AUC: 0.6237


Enhanced Training Loop with Metrics Tracking

In [55]:
from sklearn.metrics import f1_score, classification_report
import numpy as np
import matplotlib.pyplot as plt

# Initialize training history
history = {
    'train_loss': [],
    'val_auc': [],
    'val_f1': [],
    'optimal_thresholds': []
}

# Threshold optimization function
def find_optimal_threshold(y_true, y_pred_proba):
    thresholds = np.linspace(0.01, 0.5, 50)
    best_thresh = 0.5
    best_f1 = 0
    for thresh in thresholds:
        f1 = f1_score(y_true, y_pred_proba > thresh, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    return best_thresh

In [56]:
for epoch in range(3):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        inputs = {
            'input_ids': batch['input_ids'].to('cuda'),
            'attention_mask': batch['attention_mask'].to('cuda'),
            'labels': batch['labels'].to('cuda')
        }
        outputs = model(**inputs)
        loss = loss_fct(outputs.logits, inputs['labels'])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    # Validation phase
    model.eval()
    toxic_preds = []
    toxic_true = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {
                'input_ids': batch['input_ids'].to('cuda'),
                'attention_mask': batch['attention_mask'].to('cuda')
            }
            logits = model(**inputs).logits
            toxic_logits = logits[:, 0]  # First column is toxic
            toxic_preds.extend(torch.sigmoid(toxic_logits).cpu().numpy())
            toxic_true.extend(batch['labels'].cpu().numpy())

In [58]:
# Calculate metrics
optimal_thresh = find_optimal_threshold(toxic_true, toxic_preds)
auc = roc_auc_score(toxic_true, toxic_preds)
f1 = f1_score(toxic_true, np.array(toxic_preds) > optimal_thresh, zero_division=0)

# Store history
history['train_loss'].append(epoch_loss / len(train_loader))
history['val_auc'].append(auc)
history['val_f1'].append(f1)
history['optimal_thresholds'].append(optimal_thresh)

print(f"\nEpoch {epoch+1}")
print(f"Train Loss: {history['train_loss'][-1]:.4f}")
print(f"Toxic Val AUC: {auc:.4f}")
print(f"Toxic Val F1: {f1:.4f} (Optimal Threshold: {optimal_thresh:.3f})")


Epoch 3
Train Loss: 0.0565
Toxic Val AUC: 0.6008
Toxic Val F1: 0.3021 (Optimal Threshold: 0.010)


Visualization

Model eval

In [61]:
def predict_toxicity(text, model, tokenizer, threshold=None):
    """Predict toxicity for a single text input"""
    if threshold is None:
        threshold = best_thresh

    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=128,
        truncation=True,
        padding='max_length',
        return_attention_mask=True,
        return_tensors='pt'
    )

    model.eval()
    with torch.no_grad():
        inputs = {
            'input_ids': encoding['input_ids'].to('cuda'),
            'attention_mask': encoding['attention_mask'].to('cuda')
        }
        logits = model(**inputs).logits
        toxic_prob = torch.sigmoid(logits[0, 0]).item()
        all_probs = torch.sigmoid(logits).cpu().numpy()[0]

    return {
        'text': text,
        'is_toxic': toxic_prob > threshold,
        'toxic_prob': toxic_prob,
        'all_predictions': dict(zip(LABELS, all_probs)),
        'threshold': threshold
    }


In [62]:
# Example usage
sample_text = "This is an extremely offensive comment that should be flagged!"
result = predict_toxicity(sample_text, model, tokenizer)
print("\nExample Prediction:")
for k, v in result.items():
    if k != 'text':
        print(f"{k:>15}: {v}")


Example Prediction:
       is_toxic: True
     toxic_prob: 0.4349001348018646
all_predictions: {'toxic': np.float32(0.43490013), 'abusive': np.float32(0.0003276205), 'vulgar': np.float32(0.0040147714), 'menace': np.float32(0.0004681587), 'offense': np.float32(0.0014435916), 'bigotry': np.float32(0.0071212775)}
      threshold: 0.01
